# 🧠 Stage 3: In-Process NLP Narrative Classifier (TF-IDF + Platt Calibration + ONNX)
**Project**: AI Meme Coin Prediction System (Solana / pump.fun)
**Goal**: Train an in-process, zero-network-latency NLP classifier to score token narrative strength ($P_{nlp}$) directly from token name and symbol, and export it as an ONNX model for sub-5ms evaluation in Node.js.

---
### Key Architectural Decisions:
1. **No External LLM in Hot Path (Pass 1 Risk #18)**: Replaces remote HuggingFace Space round-trips (1.5s warm, 30–180s cold start) with an in-process ONNX model running inside the backend process in <5ms.
2. **Platt-Calibrated Probabilities**: Uses `CalibratedClassifierCV(method='sigmoid')` to output true calibrated probabilities $P_{nlp} \in [0, 1]$.
3. **Cold-Start Dual Training**: Blends real historical token names with a weak prior of 2,000 synthetic curated meme/utility pairs (weighted at 0.3x sample weight).
4. **Interaction Blend**: The output $P_{nlp}$ feeds into the meta-learner as an interaction boost ($P_{xgb} \times (0.80 + 0.20 \times P_{nlp})$), meaning narrative cannot resurrect an unverified rug.

In [ ]:
# Step 1: Install Required Libraries
!pip install -q scikit-learn skl2onnx onnx onnxruntime pandas numpy matplotlib seaborn

In [ ]:
# Step 2: Environment & Data Ingestion
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
try:
    from sklearn.frozen import FrozenEstimator
except ImportError:
    FrozenEstimator = None
from sklearn.metrics import roc_auc_score, brier_score_loss, precision_recall_curve, auc
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType
import onnxruntime as ort

data_dir = '../data'
synth_path = os.path.join(data_dir, 'synthetic_nlp_labels.jsonl')
hist_path = os.path.join(data_dir, 'labeled_tokens_historical.jsonl')

# Load synthetic weak prior
synth_df = pd.read_json(synth_path, lines=True)
synth_df['text'] = synth_df['name'] + ' ' + synth_df['symbol']
synth_df['weight'] = 0.3

# Load historical tokens
hist_df = pd.read_json(hist_path, lines=True)
hist_df['text'] = hist_df['mint'].apply(lambda x: f"Token {x[:6]} {x[-4:]}")
hist_df['weight'] = 1.0

df = pd.concat([
    synth_df[['text', 'label', 'weight']],
    hist_df[['text', 'label', 'weight']].sample(n=min(len(hist_df), 5000), random_state=42)
]).sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"Combined NLP training corpus: {len(df)} samples ({df['label'].sum()} positive).")

## Step 3: Train-Test Split & Vectorization
We use TF-IDF word n-grams (1, 2) with sublinear term-frequency scaling.

In [ ]:
n_train = int(len(df) * 0.80)
train_df = df.iloc[:n_train]
test_df = df.iloc[n_train:]

vec = TfidfVectorizer(ngram_range=(1, 2), max_features=2500, sublinear_tf=True)
X_train_vec = vec.fit_transform(train_df['text'])
X_test_vec = vec.transform(test_df['text'])

print(f"Vocabulary size: {len(vec.vocabulary_)} features extracted.")

## Step 4: Model Fitting & Platt Calibration
Fits Logistic Regression with sample weights (0.3x for synthetic, 1.0x for verified), then applies Platt scaling (`sigmoid`).

In [ ]:
base_lr = LogisticRegression(C=1.0, max_iter=500, random_state=42)
base_lr.fit(X_train_vec, train_df['label'], sample_weight=train_df['weight'])

if FrozenEstimator is not None:
    calibrator = CalibratedClassifierCV(estimator=FrozenEstimator(base_lr), method='sigmoid')
else:
    calibrator = CalibratedClassifierCV(estimator=base_lr, method='sigmoid', cv='prefit')

calibrator.fit(X_train_vec, train_df['label'])
print("Platt calibration fitted successfully.")

## Step 5: Out-of-Fold Evaluation
Evaluates calibration and discrimination on the held-out test split.

In [ ]:
test_probs = calibrator.predict_proba(X_test_vec)[:, 1]
test_auc = roc_auc_score(test_df['label'], test_probs)
test_brier = brier_score_loss(test_df['label'], test_probs)
p, r, _ = precision_recall_curve(test_df['label'], test_probs)
test_pr_auc = auc(r, p)

print("=== Held-Out NLP Evaluation ===")
print(f"ROC-AUC:     {test_auc:.4f}")
print(f"PR-AUC:      {test_pr_auc:.4f}")
print(f"Brier Score: {test_brier:.4f}")

# Qualitative Check on Archetype Prompts
sample_tests = [
    "Autonomous Virtual Beings AVB",
    "Goatseus Maximus GOAT",
    "test token pump 123",
    "dev buy refund coin SAFU"
]
sample_vec = vec.transform(sample_tests)
sample_scores = calibrator.predict_proba(sample_vec)[:, 1]
print("\nQualitative Checks:")
for text, score in zip(sample_tests, sample_scores):
    print(f"{text:<32} -> Score: {score:.3f}")

## Step 6: Export In-Process ONNX Pipeline
Converts the full end-to-end TF-IDF + Logistic Regression pipeline to ONNX and verifies sub-5ms execution.

In [ ]:
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)
onnx_path = os.path.join(models_dir, 'nlp_classifier.onnx')

pipeline = Pipeline([
    ('tfidf', vec),
    ('clf', base_lr)
])

initial_type = [('string_input', StringTensorType([None, 1]))]
onnx_nlp = convert_sklearn(pipeline, initial_types=initial_type, target_opset=12)

with open(onnx_path, 'wb') as f:
    f.write(onnx_nlp.SerializeToString())

print(f"Exported NLP ONNX model to {onnx_path} ({os.path.getsize(onnx_path)} bytes)")

# Verify with ONNX Runtime & Benchmark Latency
import time
session = ort.InferenceSession(onnx_path)
input_name = session.get_inputs()[0].name
test_inputs = np.array([["Autonomous Virtual Beings AVB"]], dtype=object)

times = []
for _ in range(100):
    t0 = time.perf_counter()
    session.run(None, {input_name: test_inputs})
    times.append((time.perf_counter() - t0) * 1000.0)

median_ms = np.median(times)
p99_ms = np.percentile(times, 99)
print(f"ONNX Inference Latency (100 runs): Median={median_ms:.2f}ms | p99={p99_ms:.2f}ms (Target: < 5.0ms)")